# Neural network

In [12]:
import numpy as np

In [13]:



# ============================================================
# TẬP HUẤN LUYỆN (Train) - [giờ học, tỉ lệ đi học, nhãn] (1=Đậu, 0=Rớt)
# ============================================================
train_set = [
    (8, 0.90, 1), (7, 0.80, 1), (9, 0.85, 1), (6, 0.70, 1), (10, 0.95, 1),
    (5, 0.60, 1), (7, 0.50, 1), (4, 0.90, 1),    # đi học chuyên cần bù giờ ít
    (2, 0.40, 0), (3, 0.50, 0), (1, 0.30, 0), (4, 0.45, 0), (2, 0.60, 0),
    (3, 0.35, 0), (5, 0.40, 0),
    (3, 0.40, 1),    # NGOẠI LỆ: học ít nhưng vẫn ĐẬU
    (8, 0.85, 0),    # NGOẠI LỆ: học nhiều nhưng RỚT
]

# ============================================================
# TẬP KIỂM TRA (Test) - dữ liệu MỚI, mạng CHƯA từng thấy
# ============================================================
test_set = [
    (9, 0.90, 1), (5, 0.80, 1), (4, 0.85, 1), (2, 0.45, 0),
    (3, 0.30, 0), (6, 0.55, 1), (1, 0.50, 0), (7, 0.40, 0),
]

train = np.array(train_set, dtype=float)
test  = np.array(test_set, dtype=float)
X_train, Y_train = train[:, :2], train[:, 2:3]
X_test,  Y_test  = test[:, :2],  test[:, 2:3]

def in_dataset(ten, X, Y):
    print("=" * 60)
    print(f"  {ten}")
    print("=" * 60)
    print(f"{'STT':<5}{'Giờ học':>10}{'Đi học':>10}{'Kết quả':>12}")
    print("-" * 60)
    for i in range(len(X)):
        print(f"{i+1:<5}{X[i,0]:>10.0f}{X[i,1]*100:>9.0f}%{('Đậu' if Y[i,0] else 'Rớt'):>12}")
    print(f"Tổng: {len(X)} SV | Đậu: {int(Y.sum())} | Rớt: {int(len(Y)-Y.sum())}\n")

in_dataset("TẬP HUẤN LUYỆN (Train)", X_train, Y_train)
in_dataset("TẬP KIỂM TRA (Test - dữ liệu mới)", X_test, Y_test)

# Chuẩn hóa theo max của tập train
X_max = X_train.max(axis=0)
Xtr = X_train / X_max
Xte = X_test / X_max

relu = lambda z: np.maximum(0, z)
sigmoid = lambda z: 1 / (1 + np.exp(-z))

# ============================================================
# KHỞI TẠO + HUẤN LUYỆN (1 lớp ẩn)
# ============================================================
np.random.seed(1)
W1, b1 = np.random.randn(2, 4) * 0.5, np.zeros((1, 4))   # lớp ẩn: 2 -> 4
W2, b2 = np.random.randn(4, 1) * 0.5, np.zeros((1, 1))   # lớp ra: 4 -> 1
lr, epochs = 0.3, 20000

for ep in range(epochs):
    Z1 = Xtr @ W1 + b1; A1 = relu(Z1)        # Forward
    Z2 = A1 @ W2 + b2;  A2 = sigmoid(Z2)
    dZ2 = A2 - Y_train                        # Backward
    dW2, db2 = A1.T @ dZ2 / len(Xtr), dZ2.mean(0, keepdims=True)
    dZ1 = (dZ2 @ W2.T) * (Z1 > 0)
    dW1, db1 = Xtr.T @ dZ1 / len(Xtr), dZ1.mean(0, keepdims=True)
    W1 -= lr*dW1; b1 -= lr*db1; W2 -= lr*dW2; b2 -= lr*db2   # Update

def predict(Xin):
    A1 = relu(Xin @ W1 + b1)
    return sigmoid(A1 @ W2 + b2)

# ============================================================
# BỘ THAM SỐ TỐI ƯU
# ============================================================
np.set_printoptions(precision=4, suppress=True)
print("=" * 60)
print("  BỘ THAM SỐ TỐI ƯU SAU KHI HUẤN LUYỆN")
print("=" * 60)
print("\n--- LỚP ẨN (Hidden) ---\nW1:\n", W1, "\nb1:\n", b1)
print("\n--- LỚP ĐẦU RA (Output) ---\nW2:\n", W2, "\nb2:\n", b2)

# ============================================================
# IN KẾT QUẢ DỰ ĐOÁN
# ============================================================
def in_ketqua(ten, Xraw, Yreal, prob):
    print("\n" + "=" * 60)
    print(f"  {ten}")
    print("=" * 60)
    print(f"{'STT':<5}{'Giờ học':>9}{'Đi học':>9}{'Y thực':>9}{'Xác suất':>11}{'Dự đoán':>9}{'':>4}")
    print("-" * 60)
    sai = []
    for i in range(len(Xraw)):
        p = 1 if prob[i,0] >= 0.5 else 0
        kq = "Đậu" if p else "Rớt"
        thuc = "Đậu" if Yreal[i,0] else "Rớt"
        dung = "✓" if p == int(Yreal[i,0]) else "✗"
        if p != int(Yreal[i,0]): sai.append(i)
        print(f"{i+1:<5}{Xraw[i,0]:>9.0f}{Xraw[i,1]*100:>8.0f}%{thuc:>9}{prob[i,0]:>11.4f}{kq:>9}{dung:>4}")
    acc = ((prob >= 0.5).astype(int) == Yreal).mean() * 100
    print("-" * 60)
    print(f"Accuracy: {acc:.1f}%")
    return acc, sai

acc_tr, _ = in_ketqua("KẾT QUẢ TRÊN TẬP HUẤN LUYỆN (Train)", X_train, Y_train, predict(Xtr))
acc_te, sai_te = in_ketqua("KẾT QUẢ TRÊN TẬP KIỂM TRA (Test - chưa từng thấy)", X_test, Y_test, predict(Xte))



  TẬP HUẤN LUYỆN (Train)
STT     Giờ học    Đi học     Kết quả
------------------------------------------------------------
1             8       90%         Đậu
2             7       80%         Đậu
3             9       85%         Đậu
4             6       70%         Đậu
5            10       95%         Đậu
6             5       60%         Đậu
7             7       50%         Đậu
8             4       90%         Đậu
9             2       40%         Rớt
10            3       50%         Rớt
11            1       30%         Rớt
12            4       45%         Rớt
13            2       60%         Rớt
14            3       35%         Rớt
15            5       40%         Rớt
16            3       40%         Đậu
17            8       85%         Rớt
Tổng: 17 SV | Đậu: 9 | Rớt: 8

  TẬP KIỂM TRA (Test - dữ liệu mới)
STT     Giờ học    Đi học     Kết quả
------------------------------------------------------------
1             9       90%         Đậu
2             5       80% 